In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("workspace.bronze.gho_raw")

sem_extras = df_bronze.Dim2.isNull() & df_bronze.Dim3.isNull()

filtro = (
    ((df_bronze.IndicatorCode == "GHED_GGHE-D_pc_US_SHA2011") & df_bronze.Dim1.isNull() & sem_extras) |
    ((df_bronze.IndicatorCode == "HIV_0000000026") & df_bronze.Dim1.isNull() & sem_extras) |
    ((df_bronze.IndicatorCode == "MDG_0000000029") & df_bronze.Dim1.isNull() & sem_extras) |
    ((df_bronze.IndicatorCode == "NUTRITION_ANT_HAZ_NE2") & df_bronze.Dim1.isNull() & sem_extras) |

    ((df_bronze.IndicatorCode == "M_Est_smk_curr_std") & (df_bronze.Dim1 == "SEX_BTSX") & sem_extras) |
    ((df_bronze.IndicatorCode == "WHOSIS_000001") & (df_bronze.Dim1 == "SEX_BTSX") & sem_extras) |
    ((df_bronze.IndicatorCode == "WHOSIS_000002") & (df_bronze.Dim1 == "SEX_BTSX") & sem_extras) |

    ((df_bronze.IndicatorCode == "BP_04") & (df_bronze.Dim1 == "SEX_BTSX")
     & (df_bronze.Dim2 == "AGEGROUP_YEARS30-79") & df_bronze.Dim3.isNull()) |
    ((df_bronze.IndicatorCode == "NCDMORT3070") & (df_bronze.Dim1 == "SEX_BTSX")
     & (df_bronze.Dim2 == "AGEGROUP_YEARS30-69") & df_bronze.Dim3.isNull()) |
    ((df_bronze.IndicatorCode == "NCD_BMI_30C") & (df_bronze.Dim1 == "SEX_BTSX")
     & (df_bronze.Dim2 == "AGEGROUP_YEARS18-PLUS") & df_bronze.Dim3.isNull()) |
    ((df_bronze.IndicatorCode == "NCD_GLUC_04") & (df_bronze.Dim1 == "SEX_BTSX")
     & (df_bronze.Dim2 == "AGEGROUP_YEARS18-PLUS") & df_bronze.Dim3.isNull()) |

    ((df_bronze.IndicatorCode == "NUTRITION_ANAEMIA_REPRODUCTIVEAGE_PREV")
     & (df_bronze.Dim1 == "SEX_FMLE")
     & (df_bronze.Dim2 == "PREGNANCYSTATUS_TOTAL")
     & df_bronze.Dim3.isNull()) |

    ((df_bronze.IndicatorCode == "SDGPM25") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL") & sem_extras) |
    ((df_bronze.IndicatorCode == "WSH_WATER_SAFELY_MANAGED") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL") & sem_extras) |
    ((df_bronze.IndicatorCode == "WSH_SANITATION_SAFELY_MANAGED") & (df_bronze.Dim1 == "RESIDENCEAREATYPE_TOTL") & sem_extras)
)

df_silver = (
    df_bronze
    .filter(F.col("SpatialDimType") == "COUNTRY")
    .filter(filtro)
    .select(
        F.col("SpatialDim").alias("country_code"),
        F.col("ParentLocationCode").alias("region"),
        F.col("TimeDim").alias("year"),
        F.col("IndicatorCode").alias("indicator_code"),
        F.col("NumericValue").alias("numeric_value"),
        F.col("Low").alias("low"),
        F.col("High").alias("high"),
    )
)

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.gho_clean"))

In [0]:
silver = spark.table("workspace.silver.gho_clean")

dups = silver.groupBy("country_code", "year", "indicator_code").count().filter("count > 1")
n = dups.count()
assert n == 0, f"{n} combinações duplicadas em workspace.silver.gho_clean"

presentes = silver.select("indicator_code").distinct().count()
assert presentes == 15, f"{presentes} indicadores na silver, esperado 15"

print(f"OK: {silver.count()} linhas, chave única válida, {presentes} indicadores")